# Teaching a tiny model to talk like the fishermen

The live page's chat is hand-written lines for now. Here you fine-tune a small language model, **SmolLM2-135M** (135 million weights, made by Hugging Face), to write them instead, in each player's voice.

It's the same move as the clone notebook. The clone learned to fish by copying the heuristic. This one learns to talk by copying a bigger model: last night `gemma3:12b` (12 billion weights, running in Ollama on this machine) wrote about 4,500 lines for real moments from real games. A model 90 times smaller learns to write lines like them. That's called **distillation**.

1. **Lines into tokens**: the data, and how text becomes numbers
2. **Before**: what the model says before any training
3. **Training**: start it and walk away, about 25 minutes on the CPU
4. **Before vs after vs teacher**: read them side by side

## Pocket 1: lines into tokens

Each training example is a pair:

- **the prompt**: what just happened, as text. It's the same text the browser will build: who's talking, where, the moment, the score.
- **the line**: what the teacher wrote back.

Three things before training:

- **Cap the repeats.** The teacher has favorites (in one run, "oof" showed up 162 times). Too many copies and the model learns one catchphrase, so any exact line is kept at most 3 times per speaker and kind of moment.
- **Trim the quick reactions.** The teacher wrote lots of one-to-three word reactions ("gg", "WHAT", "void."). Real chat is full of them, but too many and the model mostly says two words. So they're trimmed to about a third of chat lines and a fifth of the streamer's.
- **Hold out whole moments**, like holding out whole games in the clone notebook. All the lines for 150 moments are set aside, so pocket 4 tests on moments the model never trained on.

In [1]:
import json
import random
from collections import Counter
from pathlib import Path

import torch

random.seed(0)
torch.manual_seed(0)

rows = [json.loads(l) for l in Path("data/lines.jsonl").read_text(encoding="utf-8").splitlines()]

# any exact line at most 3 times per speaker and kind of moment, so "gg" at the end of a run
# stays in, but one voice can't say it 50 times
seen = Counter()
kept = []
for r in rows:
    key = (r["speaker"], r["slot"], r["line"].lower())
    if seen[key] < 3:
        seen[key] += 1
        kept.append(r)

# trim the quick reactions so they don't drown out real sentences: about a third of chat,
# a fifth of the streamer's speech bubble. Every longer line is kept.
SHORT_SHARE = {"chat": 0.35, "streamer": 0.20}
trimmed = []
for role, share in SHORT_SHARE.items():
    lines = [r for r in kept if r["role"] == role]
    short = [r for r in lines if len(r["line"].split()) <= 3]
    longer = [r for r in lines if len(r["line"].split()) > 3]
    n = min(len(short), round(share / (1 - share) * len(longer)))
    trimmed += longer + random.sample(short, n)
kept = trimmed

# hold out every line for 150 moments
moment_ids = sorted({r["id"] for r in kept})
random.shuffle(moment_ids)
val_ids = set(moment_ids[:150])
train = [r for r in kept if r["id"] not in val_ids]
val = [r for r in kept if r["id"] in val_ids]
print(f"{len(rows)} lines, {len(kept)} after capping and trimming: {len(train)} to train on, {len(val)} held out")

example = train[7]
print()
print(example["prompt"])
print("->", example["line"])

6590 lines, 3359 after capping and trimming: 3037 to train on, 322 held out

speaker: cast and wait (chat)
streamer: random
moment: random cast at that spot
you would have cast at: that spot
run: 0 pts, 0 caught, 0:58 left
-> that spot's a good one.


A language model doesn't read letters. It reads **tokens**: chunks of text, each one a number from its vocabulary of about 49,000. Common words are one token, and rarer ones get split into pieces.

The model is also an *instruct* model, trained to answer in a chat format with special markers for who's talking: a **system** message saying what the job is, the **user**'s message (the prompt), and the **assistant**'s answer (the line). The browser will wrap prompts in exactly the same markers.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
SYSTEM = "You write one short chat line for a pixel-art fishing stream, in the speaker's voice."

tok = AutoTokenizer.from_pretrained(NAME)


def chat_text(prompt):
    # the prompt in the model's chat format, ending where the assistant's answer starts
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    return tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


ids = tok(example["line"], add_special_tokens=False)["input_ids"]
print(f"{example['line']!r} is {len(ids)} tokens:")
print([tok.decode([i]) for i in ids])
print(ids)
print()
print(chat_text(example["prompt"]) + example["line"])

C:\Users\Jason Wong\dev\fishbowl\training\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


"that spot's a good one." is 7 tokens:
['that', ' spot', "'s", ' a', ' good', ' one', '.']
[5907, 4897, 506, 253, 1123, 582, 30]

<|im_start|>system
You write one short chat line for a pixel-art fishing stream, in the speaker's voice.<|im_end|>
<|im_start|>user
speaker: cast and wait (chat)
streamer: random
moment: random cast at that spot
you would have cast at: that spot
run: 0 pts, 0 caught, 0:58 left<|im_end|>
<|im_start|>assistant
that spot's a good one.


**Stop here for pocket 1.** You've got about 3,900 examples, and you've seen what the model actually reads: numbers, and a chat format around the prompt.

---

## Pocket 2: before

Load the model (about 270 MB, already downloaded to this machine) and ask it about a few held-out moments before any training.

**Generating** a line is the forward pass over and over: the model scores every token in its vocabulary for "what comes next", one is picked, it's added to the text, and the model goes again, until it picks the end marker. **Temperature** controls how adventurous the pick is. Near 0, it always takes the top token. At 0.8, it sometimes takes a less likely one, which is where variety comes from.

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(NAME, dtype=torch.float32).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M weights, on {device}")


@torch.no_grad()
def say(m, prompt, n=1):
    inputs = tok(chat_text(prompt), add_special_tokens=False, return_tensors="pt").to(device)
    out = m.generate(
        **inputs,
        max_new_tokens=24,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        num_return_sequences=n,
        pad_token_id=tok.pad_token_id,
    )
    start = inputs["input_ids"].shape[1]
    return [tok.decode(o[start:], skip_special_tokens=True).strip() for o in out]


# one held-out moment per few, so there's a mix of speakers and moments
probe = []
for r in val:
    if r["id"] not in {p["id"] for p in probe}:
        probe.append(r)
probe = probe[::15][:8]

model.eval()
for r in probe:
    print(r["prompt"].replace("\n", " | "))
    print("  before:", say(model, r["prompt"])[0][:90])
    print()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:  43%|████▎     | 117/272 [00:00<00:00, 1163.88it/s]

Loading weights:  86%|████████▌ | 234/272 [00:00<00:00, 1067.10it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1121.38it/s]

135M weights, on cpu
speaker: heuristic (chat) | streamer: random | moment: 25s without a catch | run: 0 pts, 0 caught, 1:35 left


...

  before: Run: 47 pts, 17 caught, 0:04 left

You cast at:

speaker: random (chat) | streamer: cast and wait | moment: 25s without a catch | run: 0 pts, 0 caught, 1:35 left


  before: Speaker: "Aren't you going to take a photo of the catch? I have the best fishing spot righ

speaker: cast and wait (chat) | streamer: the human player | moment: the stream just started | run: 0 pts, 0 caught, 2:00 left


  before: The human player cast their line and waited for a fish, but the streamer didn't catch a si

speaker: heuristic (streaming) | moment: the stream just started | run: 0 pts, 0 caught, 2:00 left


  before: Here's a revised version with some minor suggestions:

speaker: heuristic (streaming)
mome

speaker: random (streaming) | moment: the run is over: 0 pts, 0 caught


  before: "the run is over: 0 pts, 0 caught"

It's interesting to see how this line



Expect it to ramble, repeat the prompt back, or talk like a helpful assistant. It knows English and it knows chat, but it has no idea who "cast and wait" is or that it's supposed to be one.

**Stop here for pocket 2.**

---

## Pocket 3: training

Same loop as the clone, with one change: the model only gets graded on **the line**, not the prompt. Every token of the prompt is labeled `-100`, which PyTorch's loss skips, so the model learns to answer the prompt rather than to write prompts.

The loss here is how surprised the model is by each next token of the teacher's line, on average. Lower means the teacher's lines look more like what it would have said itself.

One pass over the data is about 240 steps. On this machine's CPU that's about 6 seconds a step, so **about 25 minutes**. Start it and walk away. It prints progress every 20 steps.

In [4]:
import time

BATCH = 16
EPOCHS = 1


def encode(r):
    prompt_ids = tok(chat_text(r["prompt"]), add_special_tokens=False)["input_ids"]
    answer_ids = tok(r["line"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    # grade only the answer: -100 tells the loss to skip the prompt's tokens
    return prompt_ids + answer_ids, [-100] * len(prompt_ids) + answer_ids


def batches(data, shuffle):
    order = list(range(len(data)))
    if shuffle:
        random.shuffle(order)
    for s in range(0, len(order), BATCH):
        enc = [encode(data[i]) for i in order[s : s + BATCH]]
        width = max(len(x) for x, _ in enc)  # pad every example in the batch to the longest one
        ids = torch.full((len(enc), width), tok.pad_token_id)
        labels = torch.full((len(enc), width), -100)
        mask = torch.zeros((len(enc), width), dtype=torch.long)  # 1 = real token, 0 = padding
        for j, (x, y) in enumerate(enc):
            ids[j, : len(x)] = torch.tensor(x)
            labels[j, : len(y)] = torch.tensor(y)
            mask[j, : len(x)] = 1
        yield ids.to(device), mask.to(device), labels.to(device)


@torch.no_grad()
def val_loss():
    model.eval()
    losses = [model(input_ids=i, attention_mask=m, labels=l).loss.item() for i, m, l in batches(val, False)]
    model.train()
    return sum(losses) / len(losses)


print(f"held-out loss before training: {val_loss():.2f}")
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
model.train()
t0 = time.perf_counter()
step = 0
total = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
for epoch in range(EPOCHS):
    for ids, mask, labels in batches(train, True):
        loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # no giant nudges from one odd batch
        optimizer.step()
        optimizer.zero_grad()
        step += 1
        if step % 20 == 0 or step == total:
            per = (time.perf_counter() - t0) / step
            print(f"step {step}/{total}  loss {loss.item():.2f}  about {per * (total - step) / 60:.0f} min left", flush=True)

print(f"held-out loss after training: {val_loss():.2f}")
model.save_pretrained("voice-model")
tok.save_pretrained("voice-model")
print("saved to training/voice-model/")

held-out loss before training: 4.71


step 20/190  loss 3.04  about 20 min left


step 40/190  loss 1.97  about 17 min left


step 60/190  loss 1.98  about 15 min left


step 80/190  loss 2.30  about 13 min left


...

held-out loss after training: 1.75


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.95it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]

saved to training/voice-model/


The held-out loss should drop from about 4.5 to about 2. Lower than that isn't the goal: lines like these have lots of good answers, so even a perfect model can't predict the teacher's exact words.

**Stop here for pocket 3.**

---

## Pocket 4: before vs after vs teacher

The real test is reading them. For each held-out moment: the untrained model, yours, and what the teacher wrote. Ask yourself whether you can tell who's talking without reading the name.

In [5]:
base = AutoModelForCausalLM.from_pretrained(NAME, dtype=torch.float32).to(device).eval()
model.eval()

for r in probe:
    teacher = [x["line"] for x in val if x["id"] == r["id"]]
    print(r["prompt"].replace("\n", " | "))
    print("  before: ", say(base, r["prompt"])[0][:90])
    for line in say(model, r["prompt"], n=3):
        print("  yours:  ", line)
    for line in teacher:
        print("  teacher:", line)
    print()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:  33%|███▎      | 90/272 [00:00<00:00, 899.81it/s]

Loading weights:  66%|██████▌   | 180/272 [00:00<00:00, 724.96it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 799.83it/s]

speaker: heuristic (chat) | streamer: random | moment: 25s without a catch | run: 0 pts, 0 caught, 1:35 left


...

  yours:   two minutes to go, plenty of time.
  yours:   two minutes, plenty of time.
  yours:   two minutes on the clock, plenty of time to settle in.
  teacher: just let the water do its work, friend.
  teacher: quietly waiting.

speaker: heuristic (streaming) | moment: the stream just started | run: 0 pts, 0 caught, 2:00 left


  before:  speaker: I'm really sorry for the delay, but I just can't seem to keep up with the stream.


  yours:   optimal casting angle would be 17 pixels east.
  yours:   initial conditions: zero points.
  yours:   initial conditions: zero points, two minutes.
  teacher: optimal start parameters calculated.
  teacher: expected catch rate, given run time, is 15 fish.

speaker: random (streaming) | moment: the run is over: 0 pts, 0 caught


  before:  speaker: 0
moment: 0

"I'm a fish expert and my waters are as beautiful


  yours:   still waiting!
  yours:   do they ever just go miss-placed? lol
  yours:   did i just miss something? no way
  teacher: zero points?! how is that even possible lol
  teacher: did a squirrel just run across my screen? what happened?



In [6]:
# 8 random held-out moments, different every run
by_moment = {}
for r in val:
    by_moment.setdefault(r["id"], []).append(r)

for mid in random.sample(list(by_moment), 8):
    r = by_moment[mid][0]
    print(r["prompt"].replace("\n", " | "))
    for line in say(model, r["prompt"], n=3):
        print("  yours:  ", line)
    for x in by_moment[mid]:
        print("  teacher:", x["line"])
    print()

speaker: model (chat) | streamer: random | moment: 25s without a catch | run: 0 pts, 0 caught, 1:35 left


  yours:   heuristic says...?
  yours:   still nothing!
  yours:   noted! 25 seconds isn't much time at all!
  teacher: should I recast, maybe?
  teacher: I wonder what the heuristic would prioritize now!
  teacher: was that right?

speaker: model (chat) | streamer: random | moment: a 2-pt clownfish stole the bait | run: 0 pts, 0 caught, 1:57 left


  yours:   heuristic says...
  yours:   was that right to avoid that bait?
  yours:   I wonder if I should reel in something else, maybe something else?
  teacher: dang!
  teacher: poor bait!

speaker: model (streaming) | moment: 25s without a catch | run: 7 pts, 5 caught, 1:10 left


  yours:   still waiting!
  yours:   still nothing!
  yours:   still nothing, was that right?
  teacher: was that the right timing, do you think?

speaker: random (streaming) | moment: the stream just started | run: 0 pts, 0 caught, 2:00 left


  yours:   just started lol
  yours:   i wonder if squirrels like fish lol
  yours:   hi chat, run 2:00 already? lol
  teacher: i have no plan and i love that, lol

speaker: heuristic (chat) | streamer: model | moment: model cast at the 5-pt blue tang | you would have cast at: the 5-pt blue tang (the same) | run: 3 pts, 2 caught, 1:26 left


  yours:   +2.
  yours:   5-pt tang was the optimal choice.
  yours:   5-pt tang was a better choice.
  teacher: 5-pt blue tang: logical choice.
  teacher: did you account for drift in your timing?

speaker: model (chat) | streamer: the human player | moment: the human player caught a 5-pt blue tang | run: 30 pts, 13 caught, 0:16 left


  yours:   was that right to target 30 points, given the 13?
  yours:   was that right to target the 5-pt tang?
  yours:   was that right to cast again?
  teacher: was that right to cast there?
  teacher: great catch!

speaker: cast and wait (chat) | streamer: random | moment: random reeled in nothing | run: 0 pts, 0 caught, 0:33 left


  yours:   that's fishing, isn't it?
  yours:   one point thirty-three, folks.
  yours:   that's fishing.
  teacher: just a bit early.
  teacher: thirty-three seconds left, plenty of time.

speaker: heuristic (streaming) | moment: the run is over: 45 pts, 16 caught


  yours:   optimal cast distance yielded precisely 16 catches.
  yours:   proximity.
  yours:   run score: 45. Efficiency is 60%.
  teacher: 16 catches; efficiency 72%.
  teacher: run duration clocked at 45 points, precisely.
  teacher: adequate.



Next, the trained model gets exported to the format the browser runs (ONNX) and shrunk to about 118 MB, then it goes in a web worker on the live page with Transformers.js. That part is plumbing, so it's done for you.